# ACOS Extract-Classify-ACOS — Google Colab runner

Supports **English** (`rest16`, `laptop`) and **Amharic** (`amharic`) domains.

**Pipeline overview:**
1. Mount Google Drive and install dependencies
2. Copy the repository from Drive to `/content/` (fast local SSD)
3. Patch known source-file bugs
4. XLM-RoBERTa / AfriBERTa compatibility patch
5. Settings
6. *(Amharic only)* Generate tokenized input files from the TSV data
7. Run Step 1 — aspect/opinion span extraction (BertForQuadABSA + CRF)
8. Generate candidate pairs from Step 1 predictions
9. Run Step 2 — category + sentiment classification
10. Save results back to Drive

**Before you start:**
1. Upload the entire `ACOS/` project folder to your Google Drive (e.g. `My Drive/ACOS/`).
2. In Colab: **Runtime → Change runtime type → T4 GPU**.
3. Set `DRIVE_PROJECT_PATH` in cell 1 to match where you put the folder.
4. Run all cells in order.


## 1. Mount Google Drive & install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Edit this to match where you uploaded ACOS/ in your Drive ─────────────
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/ACOS'
# ──────────────────────────────────────────────────────────────────────────

import os
if not os.path.isdir(DRIVE_PROJECT_PATH):
    raise RuntimeError(
        f'Project folder not found at {DRIVE_PROJECT_PATH}.\n'
        'Upload the ACOS/ folder to Google Drive and update DRIVE_PROJECT_PATH above.'
    )
print('Drive project found at:', DRIVE_PROJECT_PATH)


In [ ]:
!pip install --upgrade pip --quiet
!pip install torchcrf boto3 --quiet
# transformers + sentencepiece needed for XLM-RoBERTa / AfriBERTa
!pip install transformers==4.40.2 sentencepiece --quiet

import sys, subprocess, torch
print('Python:', sys.version)
try:
    print(subprocess.check_output(['nvidia-smi', '-L']).decode()[:300])
except Exception:
    print('WARNING: No GPU — enable GPU in Runtime → Change runtime type.')
print('torch.cuda.is_available():', torch.cuda.is_available())


## 2. Copy repository to working area

We copy from Drive to `/content/` so training writes go to Colab's fast local SSD.
Direct Drive writes are slow and can cause timeout errors during long training runs.

> After training finishes, run the **Save results to Drive** cell at the end
> to copy everything back before the session disconnects.


In [ ]:
import os, shutil

OUTPUT_BASE = '/content'   # all model outputs go here

repo_src = os.path.join(DRIVE_PROJECT_PATH, 'Extract-Classify-ACOS')
if not os.path.isdir(repo_src):
    raise RuntimeError(
        f'Extract-Classify-ACOS/ not found inside {DRIVE_PROJECT_PATH}.\n'
        'Make sure the full ACOS project structure was uploaded to Drive.'
    )

repo_dst = '/content/Extract-Classify-ACOS'
if os.path.exists(repo_dst):
    shutil.rmtree(repo_dst)
shutil.copytree(repo_src, repo_dst)
os.chdir(repo_dst)
print('Working directory:', os.getcwd())


## 3. Patch known source-file bugs

In [ ]:
def patch_file(path, old, new, description):
    with open(path, 'r', encoding='utf-8') as fh:
        src = fh.read()
    if old not in src:
        print(f'[SKIP] {description} — already patched')
        return
    with open(path, 'w', encoding='utf-8') as fh:
        fh.write(src.replace(old, new, 1))
    print(f'[OK]   {description}')

cwd = os.getcwd()

patch_file(
    os.path.join(cwd, 'dataset_utils.py'),
    "sys.path.insert(0, '/mnt/nfs-storage-titan/BERT/pytorch_pretrained_BERT')\n"
    "from pytorch_pretrained_bert.tokenization import BertTokenizer",
    "from bert_utils.tokenization import BertTokenizer",
    'dataset_utils.py — hard-coded NFS import'
)
patch_file(
    os.path.join(cwd, 'run_step1.py'),
    "                if args.gradient_accumulation_steps > 1:\n"
    "                    loss = loss / args.gradient_accumulation_steps\n"
    "                    ae_loss = ae_loss / args.gradient_accumulation_steps",
    "                if args.gradient_accumulation_steps > 1:\n"
    "                    loss = loss / args.gradient_accumulation_steps",
    'run_step1.py — undefined ae_loss in gradient_accumulation block'
)
patch_file(
    os.path.join(cwd, 'run_step1.py'),
    "                    optimizer.backward(ae_loss)",
    "                    optimizer.backward(loss)",
    'run_step1.py — fp16 backward used ae_loss instead of loss'
)
for old_call, desc in [
    ("eval_features = convert_examples_to_features2nd(\n"
     "            eval_examples, label_list, args.max_seq_length, tokenizer, task_name)",
     'run_step2.py — eval features'),
    ("train_features = convert_examples_to_features2nd(\n"
     "        train_examples, label_list, args.max_seq_length, tokenizer, task_name)",
     'run_step2.py — train features'),
    ("valid_features = convert_examples_to_features2nd(\n"
     "        valid_examples, label_list, args.max_seq_length, tokenizer, task_name)",
     'run_step2.py — valid features'),
]:
    patch_file(os.path.join(cwd, 'run_step2.py'), old_call,
               old_call.replace(', task_name)', ', output_mode="classification")'), desc)

print('\nAll patches done.')


## 4. XLM-RoBERTa / AfriBERTa compatibility patch

The codebase uses a standalone BERT stack (`bert_utils/`). XLM-RoBERTa and
AfriBERTa need a SentencePiece tokenizer and a different model format.
This cell monkey-patches two things at runtime without touching source files:

1. `bert_utils.tokenization.BertTokenizer` → thin `AutoTokenizer` shim
2. `modeling.BertPreTrainedModel.from_pretrained` → transplants XLM-R encoder weights

Both patches are **no-ops for plain `bert-*` models** so English domains are unaffected.


In [ ]:
import os, sys

_repo = '/content/Extract-Classify-ACOS'
if _repo not in sys.path:
    sys.path.insert(0, _repo)

def _is_bert_model(name):
    base = os.path.basename(name.rstrip('/')).lower()
    return base.startswith('bert-') or base.startswith('bert_')

import bert_utils.tokenization as _tok_module
_OrigBertTokenizer = _tok_module.BertTokenizer

class _AutoTokenizerShim:
    """
    Minimal shim that exposes the BertTokenizer API expected by the pipeline
    while delegating to transformers.AutoTokenizer under the hood.
    """
    # BERT special-token strings mapped to AutoTokenizer attribute names
    _SPECIAL = {'[CLS]': 'cls_token', '[SEP]': 'sep_token', '[PAD]': 'pad_token'}

    def __init__(self, hf_tok):
        self._tok = hf_tok
        self.vocab = {v: k for k, v in hf_tok.get_vocab().items()}
        self.ids_to_tokens = {k: v for v, k in self.vocab.items()}

    @classmethod
    def from_pretrained(cls, name, do_lower_case=False, **kw):
        if _is_bert_model(name):
            return _OrigBertTokenizer.from_pretrained(name, do_lower_case=do_lower_case, **kw)
        from transformers import AutoTokenizer
        return cls(AutoTokenizer.from_pretrained(name, use_fast=True))

    def _special_id(self, bert_name):
        """Map [CLS]/[SEP]/[PAD] to the model's own special-token ID."""
        attr = self._SPECIAL.get(bert_name)
        if attr:
            special_str = getattr(self._tok, attr, None)
            if special_str is not None:
                return self._tok.get_vocab().get(special_str, 0)
        return self.vocab.get(bert_name, 0)

    def convert_tokens_to_ids(self, tokens):
        """Convert token strings to IDs, mapping BERT special tokens to model equivalents."""
        result = []
        for tok in tokens:
            if tok in self._SPECIAL:
                result.append(self._special_id(tok))
            else:
                # Try direct vocab lookup first (works for BERT whole-word vocab)
                direct = self.vocab.get(tok)
                if direct is not None:
                    result.append(direct)
                else:
                    # SentencePiece (XLM-R): tokenize the word, take the first piece ID
                    pieces = self._tok.tokenize(tok)
                    if pieces:
                        result.append(self._tok.convert_tokens_to_ids(pieces)[0])
                    else:
                        result.append(getattr(self._tok, 'unk_token_id', 3))
        return result

    def convert_ids_to_tokens(self, ids):
        return self._tok.convert_ids_to_tokens(ids)

    def tokenize(self, text):
        return self._tok.tokenize(text)

    def save_vocabulary(self, save_dir, filename_prefix=None):
        """Save the tokenizer vocab/config so the model directory is self-contained."""
        import os
        os.makedirs(save_dir, exist_ok=True)
        # save_pretrained writes tokenizer.json, tokenizer_config.json, etc.
        saved = self._tok.save_pretrained(save_dir)
        print(f'[shim] Tokenizer saved to {save_dir} ({len(saved)} files)')
        # Return a dummy vocab file path so callers that expect a string don't crash
        return (os.path.join(save_dir, 'tokenizer.json'),)

    def __call__(self, text, **kw):
        return self._tok(text, **kw)

_tok_module.BertTokenizer = _AutoTokenizerShim
print('[patch] BertTokenizer -> _AutoTokenizerShim')

import modeling as _mod
_Orig = _mod.BertPreTrainedModel.from_pretrained.__func__

@classmethod
def _patched(cls, name, *inputs, **kwargs):
    import torch
    from transformers import AutoModel, AutoConfig
    import modeling as _m
    if _is_bert_model(name):
        return _Orig(cls, name, *inputs, **kwargs)
    print(f'[patch] Loading {name} via transformers.AutoModel ...')
    hf_cfg = AutoConfig.from_pretrained(name)
    hf_enc = AutoModel.from_pretrained(name)
    bert_cfg = _m.BertConfig(
        vocab_size_or_config_json_file=hf_cfg.vocab_size,
        hidden_size=hf_cfg.hidden_size,
        num_hidden_layers=hf_cfg.num_hidden_layers,
        num_attention_heads=hf_cfg.num_attention_heads,
        intermediate_size=hf_cfg.intermediate_size,
        hidden_act=getattr(hf_cfg,'hidden_act','gelu'),
        hidden_dropout_prob=hf_cfg.hidden_dropout_prob,
        attention_probs_dropout_prob=hf_cfg.attention_probs_dropout_prob,
        max_position_embeddings=getattr(hf_cfg,'max_position_embeddings',514),
        type_vocab_size=getattr(hf_cfg,'type_vocab_size',1),
        initializer_range=getattr(hf_cfg,'initializer_range',0.02),
    )
    model = cls(bert_cfg, num_labels=kwargs.get('num_labels',2))
    hf_sd = hf_enc.state_dict(); own_sd = model.state_dict(); remapped = {}
    for k, v in hf_sd.items():
        nk = 'bert.' + k[k.index('.')+1:] if '.' in k else 'bert.' + k
        for pfx in ('roberta.','bert.'):
            if k.startswith(pfx): nk = 'bert.' + k[len(pfx):]; break
        if nk in own_sd and own_sd[nk].shape == v.shape: remapped[nk] = v
    print(f'[patch] Transplanted {len(remapped)}/{sum(1 for k in own_sd if "bert." in k)} tensors.')
    own_sd.update(remapped); model.load_state_dict(own_sd)
    return model

_mod.BertPreTrainedModel.from_pretrained = _patched
print('[patch] BertPreTrainedModel.from_pretrained -> _patched')
print('XLM-R / AfriBERTa compatibility patches applied.')


## 5. Settings

Set `DOMAIN` to one of:
- `'rest16'` — English restaurant reviews (SemEval-2016)
- `'laptop'` — English laptop reviews (SemEval-2016)
- `'amharic'` — Amharic public-affairs reviews (CAC_DA dataset)

For `'amharic'` the model defaults to `xlm-roberta-base`.
Switch to `'castorini/afriberta_large'` for a smaller Africa-focused model,
or `'bert-base-multilingual-cased'` if you hit GPU memory limits.


In [ ]:
# ── User-editable settings ──────────────────────────────────────────────
DOMAIN = 'amharic'   # 'rest16' | 'laptop' | 'amharic'

EPOCHS_STEP1           = 10
EPOCHS_STEP2           = 10
TRAIN_BATCH_SIZE_STEP1 = 8
TRAIN_BATCH_SIZE_STEP2 = 8
MAX_SEQ_LENGTH         = 128
LEARNING_RATE_STEP1    = 2e-5
LEARNING_RATE_STEP2    = 5e-5

# ── Domain-specific overrides ────────────────────────────────────────────
if DOMAIN == 'amharic':
    BERT_MODEL    = 'xlm-roberta-base'
    # Alternatives:
    #   'castorini/afriberta_large'    — smaller, Africa-focused
    #   'bert-base-multilingual-cased' — mBERT fallback
    DO_LOWER_CASE = False
else:
    BERT_MODEL    = 'bert-base-uncased'
    DO_LOWER_CASE = True

lower_flag = '--do_lower_case' if DO_LOWER_CASE else ''
print(f'Domain    : {DOMAIN}')
print(f'Model     : {BERT_MODEL}')
print(f'Lowercase : {DO_LOWER_CASE}')
print(f'Output    : {OUTPUT_BASE}')


## 6. Amharic data preparation *(skip if DOMAIN != 'amharic')*

Copies the pre-built TSV files from Drive into the working repo and runs
`prepare_amharic.py` to generate the `tokenized_data/` files the pipeline needs.

The TSV files (`amharic_quad_{train,dev,test}.tsv`) were produced by
`data/Amharic-ACOS/csv_to_acos_tsv.py` from the CAC_DA source CSV.
They must be present inside the `data/Amharic-ACOS/` folder in your Drive upload.


In [ ]:
import shutil, subprocess

if DOMAIN == 'amharic':
    cwd = os.getcwd()

    # Locate Amharic data — it lives alongside Extract-Classify-ACOS in the project
    amharic_data_src = os.path.normpath(
        os.path.join(DRIVE_PROJECT_PATH, 'data', 'Amharic-ACOS')
    )
    if not os.path.isdir(amharic_data_src):
        raise RuntimeError(
            f'Amharic data not found at {amharic_data_src}.\n'
            'Make sure data/Amharic-ACOS/ is inside your Drive ACOS folder.'
        )
    print('Amharic data found at:', amharic_data_src)

    # Copy into working area
    working_data_dir = os.path.join(cwd, 'amharic_data')
    if os.path.exists(working_data_dir): shutil.rmtree(working_data_dir)
    shutil.copytree(amharic_data_src, working_data_dir)
    print('Copied to:', working_data_dir)

    # Safety-patch the label list in case an older file was copied
    utils_path = os.path.join(cwd, 'run_classifier_dataset_utils.py')
    with open(utils_path, encoding='utf-8') as fh: src = fh.read()
    NEW_LABELS = '''
        elif domain_type == \'amharic\':
            l = [
                \'ECONOMY#COMMUNITY_SUPPORT\', \'ECONOMY#EMPLOYMENT\',
                \'ECONOMY#TAXATION\', \'ECONOMY#UTILITIES\',
                \'GOVERNANCE#CITIZEN_ENGAGEMENT\', \'GOVERNANCE#COMMUNITY_SUPPORT\',
                \'GOVERNANCE#LEGISLATION\', \'GOVERNANCE#TRANSPARENCY\',
                \'HEALTHCARE#GENERAL\',
                \'INFRASTRUCTURE#TRANSPORTATION\', \'INFRASTRUCTURE#UTILITIES\',
                \'PUBLIC_SAFETY#CRIME\', \'PUBLIC_SAFETY#CRIME_SERVICES\',
                \'PUBLIC_SAFETY#EMERGENCY_SERVICES\',
                \'PUBLIC_SERVICES#COMMUNITY_SUPPORT\', \'PUBLIC_SERVICES#EDUCATION\',
                \'PUBLIC_SERVICES#EMERGENCY_SERVICES\', \'PUBLIC_SERVICES#HEALTHCARE\',
                \'PUBLIC_SERVICES#INFRASTRUCTURE\', \'PUBLIC_SERVICES#UTILITIES\',
                \'SOCIAL#COMMUNITY_SUPPORT\', \'SOCIAL#EDUCATION\',
                \'SOCIAL#EQUALITY_JUSTICE\',
            ]
        elif domain_type == \'laptop\':
'''
    if "domain_type == 'amharic'" not in src:
        src = src.replace("        elif domain_type == 'laptop':", NEW_LABELS)
        with open(utils_path, 'w', encoding='utf-8') as fh: fh.write(src)
        print('[OK]   run_classifier_dataset_utils.py — amharic labels patched')
    else:
        print('[SKIP] run_classifier_dataset_utils.py — amharic labels already present')

    # Run prepare_amharic.py
    cmd = (f'python tokenized_data/prepare_amharic.py '
           f'--data_dir {working_data_dir} --out_dir tokenized_data '
           f'--bert_model {BERT_MODEL}')
    print('\nRunning:', cmd)
    proc = subprocess.run(cmd, shell=True, check=False)
    if proc.returncode != 0:
        raise RuntimeError(f'prepare_amharic.py failed (exit {proc.returncode})')

    expected = ['tokenized_data/amharic_train_quad_bert.tsv',
                'tokenized_data/amharic_dev_quad_bert.tsv',
                'tokenized_data/amharic_test_quad_bert.tsv',
                'tokenized_data/amharic_train_pair.tsv',
                'tokenized_data/amharic_dev_pair.tsv']
    missing = [f for f in expected if not os.path.exists(f)]
    if missing: raise RuntimeError('Missing files:\n  ' + '\n  '.join(missing))
    print('\nAll tokenized data files present. Ready to train.')
else:
    print(f'DOMAIN={DOMAIN}: skipping Amharic data prep.')


## 6b. Pre-flight check *(run before Step 1)*

Runs a quick smoke test **inside the notebook process** so any import error or
tokenizer crash shows a full traceback here rather than a silent exit-code 1
inside the subprocess.  All checks must print `OK` before running Step 1.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())   # /content/Extract-Classify-ACOS

print('--- import checks ---')
import torch;            print('torch          OK', torch.__version__)
import torchcrf;         print('torchcrf       OK')
import transformers;     print('transformers   OK', transformers.__version__)

print('\n--- tokenizer ---')
import bert_utils.tokenization as _t
tok = _t.BertTokenizer.from_pretrained(BERT_MODEL)
print('Tokenizer class:', type(tok).__name__)

# Test BERT special token mapping
cls_id = tok.convert_tokens_to_ids(['[CLS]'])[0]
sep_id = tok.convert_tokens_to_ids(['[SEP]'])[0]
print(f'[CLS] id: {cls_id}   [SEP] id: {sep_id}')
assert cls_id != 3, f'[CLS] mapped to unk (id=3) — special token fix not working'
assert sep_id != 3, f'[SEP] mapped to unk (id=3) — special token fix not working'
print('Special token IDs: OK')

# Test actual Amharic word encoding (real pipeline input)
amh_word = 'ምግቡ'
amh_id = tok.convert_tokens_to_ids([amh_word])[0]
print(f'Amharic word {repr(amh_word)} -> id {amh_id}')
assert amh_id != 3, f'{repr(amh_word)} still mapping to unk — SentencePiece encoding broken'
print('Amharic word encoding: OK')

print('\n--- save_vocabulary ---')
import tempfile
with tempfile.TemporaryDirectory() as td:
    tok.save_vocabulary(td)
    saved = os.listdir(td)
print('saved files:', saved)
assert len(saved) > 0, 'save_vocabulary wrote no files'
print('save_vocabulary: OK')

print('\n--- run_classifier_dataset_utils ---')
from run_classifier_dataset_utils import _tokens_to_ids, _get_special_token_id, processors
test_tokens = ['[CLS]', amh_word, '[SEP]']
ids = _tokens_to_ids(tok, test_tokens)
print(f'_tokens_to_ids({test_tokens}) -> {ids}')
assert ids[0] == cls_id, f'[CLS] id mismatch: got {ids[0]}, expected {cls_id}'
assert ids[1] != 3,      f'Amharic word still mapping to unk after fix'
assert ids[2] == sep_id, f'[SEP] id mismatch: got {ids[2]}, expected {sep_id}'
print('_tokens_to_ids: OK')

print('\n--- eval_metrics ---')
from eval_metrics import _decode_token_ids
decoded = _decode_token_ids(tok, ids)
print(f'_decode_token_ids({ids}) -> {repr(decoded)}')
assert decoded.strip() != '', '_decode_token_ids returned empty string — decode broken'
print('_decode_token_ids: OK')

print('\n--- label list ---')
p = processors['quad']()
labels = p.get_labels(DOMAIN)
print('seq labels:', labels[1])
assert '[CLS]' in labels[1], 'seq label list missing [CLS]'
print('label list: OK')

print('\n\u2713 All pre-flight checks passed \u2014 safe to run Step 1.')


## 7. Step 1 — Aspect & Opinion Span Extraction

In [ ]:
import subprocess, sys
step1_out = os.path.join(OUTPUT_BASE,'output','Extract-Classify-QUAD',DOMAIN+'_1st')
log_file = os.path.join(OUTPUT_BASE, 'step1.log')
cmd = (
    f'python run_step1.py --task_name quad --do_train --do_eval '
    f'--domain_type {DOMAIN} --model_type quad {lower_flag} '
    f'--data_dir . --bert_model {BERT_MODEL} '
    f'--max_seq_length {MAX_SEQ_LENGTH} '
    f'--train_batch_size {TRAIN_BATCH_SIZE_STEP1} '
    f'--learning_rate {LEARNING_RATE_STEP1} '
    f'--num_train_epochs {EPOCHS_STEP1} '
    f'--output_dir {step1_out}'
    f' 2>&1 | tee {log_file}'
)
print('Running Step 1:\n', cmd)
ret = os.system(cmd)
print('Step 1 exit code:', ret)
if ret != 0:
    print('\n=== STEP 1 LAST 100 LOG LINES ===')
    with open(log_file, encoding='utf-8', errors='replace') as f:
        lines = f.readlines()
    print(''.join(lines[-100:]))


## 8. Generate Candidate Pairs from Step 1 Predictions

In [ ]:
cmd = f'python tokenized_data/get_1st_pairs.py . {DOMAIN}'
print('Running:', cmd)
proc = subprocess.run(cmd, shell=True, check=False)
print('Exit code:', proc.returncode)
pair_file = os.path.join('tokenized_data', DOMAIN + '_test_pair_1st.tsv')
if os.path.exists(pair_file):
    print(f'Generated {len(open(pair_file,encoding="utf-8").readlines())} pairs -> {pair_file}')
else:
    print(f'WARNING: {pair_file} not found — check Step 1 output.')


## 9. Step 2 — Category & Sentiment Classification

In [ ]:
step2_out = os.path.join(OUTPUT_BASE,'output','Extract-Classify-QUAD',DOMAIN+'_2nd')
log_file2 = os.path.join(OUTPUT_BASE, 'step2.log')
cmd = (
    f'python run_step2.py --task_name categorysenti --do_train --do_eval '
    f'--domain_type {DOMAIN} --model_type categorysenti {lower_flag} '
    f'--data_dir . --bert_model {BERT_MODEL} '
    f'--max_seq_length {MAX_SEQ_LENGTH} '
    f'--train_batch_size {TRAIN_BATCH_SIZE_STEP2} '
    f'--learning_rate {LEARNING_RATE_STEP2} '
    f'--num_train_epochs {EPOCHS_STEP2} '
    f'--output_dir {step2_out}'
    f' 2>&1 | tee {log_file2}'
)
print('Running Step 2:\n', cmd)
ret = os.system(cmd)
print('Step 2 exit code:', ret)
if ret != 0:
    print('\n=== STEP 2 LAST 100 LOG LINES ===')
    with open(log_file2, encoding='utf-8', errors='replace') as f:
        lines = f.readlines()
    print(''.join(lines[-100:]))


## 10. Results

In [ ]:
for step_dir, label in [
    (os.path.join(OUTPUT_BASE,'output','Extract-Classify-QUAD',DOMAIN+'_1st'),'Step 1'),
    (os.path.join(OUTPUT_BASE,'output','Extract-Classify-QUAD',DOMAIN+'_2nd'),'Step 2'),
]:
    rf = os.path.join(step_dir, 'Test_results.txt')
    if os.path.exists(rf):
        print(f'\n=== {label} Test Results ===')
        print(open(rf, encoding='utf-8').read())
    else:
        print(f'\n=== {label}: Test_results.txt not found at {rf} ===')


## 11. Save results to Drive

Colab's `/content/` is wiped when the session ends. Run this cell to copy
the output models and results back to your Drive before disconnecting.


In [ ]:
import shutil, datetime

src_outputs = os.path.join(OUTPUT_BASE, 'output', 'Extract-Classify-QUAD')
timestamp   = datetime.datetime.now().strftime('%Y%m%d_%H%M')
dst_outputs = os.path.join(DRIVE_PROJECT_PATH, f'outputs_{timestamp}')

if os.path.isdir(src_outputs):
    shutil.copytree(src_outputs, dst_outputs)
    print(f'Results saved to Drive: {dst_outputs}')
else:
    print('No output directory found — run Steps 7–9 first.')


## Notes

### Colab-specific
- Free Colab provides a T4 GPU (~16 GB VRAM). `xlm-roberta-base` fits comfortably.
  `xlm-roberta-large` (570 M params) will OOM — stick with base or AfriBERTa.
- Free Colab sessions disconnect after ~12 hours of inactivity.
  With 10 epochs on the Amharic dataset (~39 k sentences) expect ~4–8 hours per step.
  Run the two steps in separate sessions if needed, saving checkpoints to Drive in between.
- If the session disconnects mid-run, re-run cells 1–5 (mount, copy, patches) then
  resume from whichever step was interrupted.

### Amharic-specific
- `xlm-roberta-base` is the default. Alternatives: `castorini/afriberta_large` (smaller/faster),
  `bert-base-multilingual-cased` (mBERT, use if memory is tight).
- Categories: 23 `DOMAIN#ATTRIBUTE` labels from the CAC_DA public-affairs corpus.
  Defined in `run_classifier_dataset_utils.py` under `domain_type == 'amharic'`.
- To retrain on updated data: re-run `csv_to_acos_tsv.py` locally, re-upload the TSVs
  to Drive, then restart from cell 6.
